# 🎥 UCF-Crime Anomaly Detection — ViT Feature Training

**Architecture**: Temporal Transformer over ViT-Base (768-d) frame features  
**Task**: Binary anomaly classification (Normal vs Anomaly) at video level  
**Input**: `.npy` files of shape `(num_frames, 768)` — one per video

---
### Folder structure expected:
```
features/
  Fighting/
    Fighting001_x264.npy
    ...
  Normal/
    Normal001_x264.npy
    ...
  Abuse/
    ...
  (all other UCF-Crime categories)
```
Normal videos → label 0  
All other categories → label 1 (anomaly)

# Compatibility Notes

This edited copy is updated for the organized project layout and current laptop environment.

- Original notebooks remain unchanged in `notebooks/`.
- PyTorch 2.6+ checkpoint loading is handled with `weights_only=False` for trusted local checkpoints.
- VideoMAE preprocessing uses `AutoImageProcessor` instead of the deprecated/removed `VideoMAEFeatureExtractor`.
- Paths resolve from the project root so the notebook works even when opened from this subfolder.


In [ ]:
from pathlib import Path

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "artifacts").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing artifacts/ and src/")

PROJECT_ROOT = find_project_root()
print(f"Project root: {PROJECT_ROOT}")


## Cell 1 — Install Dependencies

In [ ]:
# Run once — skip if already installed
import subprocess, sys
pkgs = ['torch', 'torchvision', 'scikit-learn', 'matplotlib', 'seaborn',
        'tqdm', 'numpy', 'pandas']
for p in pkgs:
    try:
        __import__(p.replace('-', '_'))
        print(f'✅ {p} already installed')
    except ImportError:
        print(f'📦 Installing {p}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', p])
        print(f'✅ {p} installed')
print('\n🎉 All dependencies ready!')

## Cell 2 — Imports & Config

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# ⚙️  CONFIG — edit these paths and hyperparams
# ─────────────────────────────────────────────
FEATURES_ROOT = str(PROJECT_ROOT / 'data' / 'UCF-Crime_dataset' / 'VideoMAE_features')  # Root folder containing category subfolders
NORMAL_FOLDERS = ['Normal', 'Normal1']  # Folder names treated as NORMAL (label=0)

# Model
FEAT_DIM      = 768    # ViT-Base feature dimension
NUM_HEADS     = 8      # Transformer attention heads
NUM_LAYERS    = 4      # Transformer encoder layers
FF_DIM        = 1024   # Feed-forward hidden dim
DROPOUT       = 0.3    # Dropout rate
MAX_FRAMES    = 512    # Clip/pad videos to this many frames (memory control)

# Training
EPOCHS        = 50
BATCH_SIZE    = 16
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 10     # Early stopping patience
SEED          = 42

# Paths
CHECKPOINT    = str(PROJECT_ROOT / 'artifacts' / 'checkpoints' / 'best_model_training_copy.pt')
RESULTS_DIR   = str(PROJECT_ROOT / 'artifacts' / 'reports' / 'training_notebook')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('=' * 50)
print(f'  Device     : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU        : {torch.cuda.get_device_name(0)}')
print(f'  Feat dim   : {FEAT_DIM}')
print(f'  Epochs     : {EPOCHS}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Max frames : {MAX_FRAMES}')
print('=' * 50)

## Cell 3 — Scan Feature Files & Build Dataset Manifest

In [ ]:
def build_manifest(root, normal_folders):
    """Walk root directory. Normal folders → label 0, everything else → label 1."""
    root = Path(root)
    records = []
    missing = []

    if not root.exists():
        raise FileNotFoundError(
            f"\n❌ Features root '{root}' not found.\n"
            f"   Please set FEATURES_ROOT in Cell 2 to your extracted features folder."
        )

    for category_dir in sorted(root.iterdir()):
        if not category_dir.is_dir():
            continue
        label = 0 if category_dir.name in normal_folders else 1
        npy_files = sorted(category_dir.glob('*.npy'))
        if len(npy_files) == 0:
            missing.append(category_dir.name)
        for f in npy_files:
            records.append({'path': str(f), 'category': category_dir.name, 'label': label})

    if missing:
        print(f'⚠️  Empty/missing .npy in folders: {missing}')

    df = pd.DataFrame(records)
    return df


manifest = build_manifest(FEATURES_ROOT, NORMAL_FOLDERS)

print(f'\n📂 Total videos found : {len(manifest)}')
print(f'   Normal  (label=0)  : {(manifest.label==0).sum()}')
print(f'   Anomaly (label=1)  : {(manifest.label==1).sum()}')
print(f'\n📊 Per-category breakdown:')
cat_summary = manifest.groupby(['category', 'label']).size().reset_index(name='count')
print(cat_summary.to_string(index=False))

# Quick sanity-check one file
sample = np.load(manifest.iloc[0]['path'])
print(f'\n🔍 Sample file shape : {sample.shape}  (frames × feat_dim)')
assert sample.shape[1] == FEAT_DIM, \
    f'❌ Feature dim mismatch! File has {sample.shape[1]}, FEAT_DIM={FEAT_DIM}'

## Cell 4 — Dataset Class

In [ ]:
class VideoFeatureDataset(Dataset):
    """
    Loads pre-extracted ViT features from .npy files.
    Each file: (num_frames, 768).  We clip/pad to MAX_FRAMES.
    """
    def __init__(self, df, max_frames=512, augment=False):
        self.df        = df.reset_index(drop=True)
        self.max_frames = max_frames
        self.augment   = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        feat = np.load(row['path']).astype(np.float32)  # (T, 768)
        T    = feat.shape[0]

        # ── Clip or pad to MAX_FRAMES ──
        if T >= self.max_frames:
            # Random crop during training, center crop otherwise
            if self.augment:
                start = random.randint(0, T - self.max_frames)
            else:
                start = (T - self.max_frames) // 2
            feat = feat[start : start + self.max_frames]
        else:
            pad  = np.zeros((self.max_frames - T, feat.shape[1]), dtype=np.float32)
            feat = np.concatenate([feat, pad], axis=0)

        # ── Augmentation: random temporal flip & Gaussian noise ──
        if self.augment:
            if random.random() < 0.5:
                feat = feat[::-1].copy()           # Temporal flip
            if random.random() < 0.3:
                feat += np.random.normal(0, 0.02, feat.shape).astype(np.float32)  # Noise

        feat  = torch.from_numpy(feat)             # (MAX_FRAMES, 768)
        label = torch.tensor(row['label'], dtype=torch.long)
        return feat, label


print('✅ VideoFeatureDataset defined.')

## Cell 5 — Train / Val / Test Split + DataLoaders

In [ ]:
# Stratified split: 70% train, 15% val, 15% test
train_df, temp_df = train_test_split(
    manifest, test_size=0.30, stratify=manifest['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df,  test_size=0.50, stratify=temp_df['label'],   random_state=SEED
)

print(f'Train : {len(train_df)} videos  (normal={( train_df.label==0).sum()}, anomaly={( train_df.label==1).sum()})')
print(f'Val   : {len(val_df)}   videos  (normal={( val_df.label==0).sum()}, anomaly={(   val_df.label==1).sum()})')
print(f'Test  : {len(test_df)}  videos  (normal={( test_df.label==0).sum()}, anomaly={( test_df.label==1).sum()})')

# ── Weighted sampler to handle class imbalance ──
labels_train = train_df['label'].values
class_counts = np.bincount(labels_train)
class_weights = 1.0 / class_counts
sample_weights = class_weights[labels_train]
sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(train_df),
    replacement=True
)

train_ds = VideoFeatureDataset(train_df, MAX_FRAMES, augment=True)
val_ds   = VideoFeatureDataset(val_df,   MAX_FRAMES, augment=False)
test_ds  = VideoFeatureDataset(test_df,  MAX_FRAMES, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,   num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,     num_workers=0, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,     num_workers=0, pin_memory=True)

print(f'\n🔄 Train batches : {len(train_loader)}')
print(f'   Val   batches : {len(val_loader)}')
print(f'   Test  batches : {len(test_loader)}')

## Cell 6 — Model: Temporal Transformer Anomaly Detector

In [ ]:
class PositionalEncoding(nn.Module):
    """Learnable positional encoding."""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.pe = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        T = x.size(1)
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)
        return x + self.pe(pos)                               # (B, T, D)


class AnomalyTransformer(nn.Module):
    """
    Temporal Transformer for video-level anomaly classification.

    1. Linear projection: 768 → d_model
    2. Learnable positional encoding
    3. N × Transformer encoder layers
    4. Attention-pooled [CLS] token output
    5. MLP head → 2-class logits
    """
    def __init__(self, feat_dim=768, d_model=512, num_heads=8,
                 num_layers=4, ff_dim=1024, dropout=0.3, max_frames=512):
        super().__init__()

        self.proj = nn.Sequential(
            nn.Linear(feat_dim, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos_enc   = PositionalEncoding(d_model, max_len=max_frames + 1)  # +1 for CLS

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers,
            norm=nn.LayerNorm(d_model)
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 2)
        )

    def forward(self, x):
        # x: (B, T, 768)
        B = x.size(0)
        x = self.proj(x)                                        # (B, T, d_model)
        cls = self.cls_token.expand(B, -1, -1)                  # (B, 1, d_model)
        x   = torch.cat([cls, x], dim=1)                        # (B, T+1, d_model)
        x   = self.pos_enc(x)                                   # positional encoding
        x   = self.transformer(x)                               # (B, T+1, d_model)
        cls_out = x[:, 0, :]                                    # CLS token
        logits  = self.head(cls_out)                            # (B, 2)
        return logits


model = AnomalyTransformer(
    feat_dim=FEAT_DIM, d_model=512, num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS, ff_dim=FF_DIM, dropout=DROPOUT, max_frames=MAX_FRAMES
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\n🧠 Model: AnomalyTransformer')
print(f'   Total params    : {total_params:,}')
print(f'   Trainable params: {trainable:,}')

# Quick forward test
dummy = torch.randn(2, MAX_FRAMES, FEAT_DIM).to(DEVICE)
out   = model(dummy)
print(f'   Output shape    : {out.shape}  ✅')

## Cell 7 — Loss, Optimizer, Scheduler

In [ ]:
# Class-weighted loss to handle imbalance
n_normal  = (train_df.label == 0).sum()
n_anomaly = (train_df.label == 1).sum()
total     = n_normal + n_anomaly
w_normal  = total / (2 * n_normal)  if n_normal  > 0 else 1.0
w_anomaly = total / (2 * n_anomaly) if n_anomaly > 0 else 1.0
class_wt  = torch.tensor([w_normal, w_anomaly], dtype=torch.float32).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_wt, label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

print(f'✅ Loss     : CrossEntropyLoss (label_smoothing=0.1, class_weights=[{w_normal:.3f}, {w_anomaly:.3f}])')
print(f'   Optimizer: AdamW  (lr={LR}, weight_decay={WEIGHT_DECAY})')
print(f'   Scheduler: CosineAnnealingLR (T_max={EPOCHS})')

## Cell 8 — Training & Validation Loop

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, device=DEVICE, desc='Train'):
    """One epoch. If optimizer is None → validation mode."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for feats, labels in loader:
            feats, labels = feats.to(device), labels.to(device)

            logits = model(feats)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            probs  = F.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
            preds  = logits.argmax(dim=1)
            total_loss += loss.item() * labels.size(0)
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)
            all_probs.extend(probs)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / total
    acc      = correct / total * 100
    try:
        auc = roc_auc_score(all_labels, all_probs) * 100
    except Exception:
        auc = 0.0

    return avg_loss, acc, auc, np.array(all_labels), np.array(all_probs)


# ── Training history ──
history = {
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'train_auc':  [], 'val_auc':  [],
    'lr':         []
}

best_val_auc  = -1
patience_cnt  = 0

print('🚀 Starting Training')
print('=' * 85)
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>10} | {"Train AUC":>10} | {"Val Loss":>8} | {"Val Acc":>8} | {"Val AUC":>8} | LR')
print('─' * 85)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, tr_auc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_auc, _, _ = run_epoch(model, val_loader,   criterion)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    history['train_auc'].append(tr_auc)
    history['val_auc'].append(vl_auc)
    history['lr'].append(lr)

    marker = ''
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer_state': optimizer.state_dict(),
                    'val_auc': vl_auc, 'val_acc': vl_acc}, CHECKPOINT)
        patience_cnt = 0
        marker = '  ⭐ best'
    else:
        patience_cnt += 1

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {tr_acc:>9.2f}% | {tr_auc:>9.2f}% | '
          f'{vl_loss:>8.4f} | {vl_acc:>7.2f}% | {vl_auc:>7.2f}%{marker}')

    if patience_cnt >= PATIENCE:
        print(f'\n🛑 Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

print('─' * 85)
print(f'✅ Training complete. Best Val AUC = {best_val_auc:.2f}%  (checkpoint saved to {CHECKPOINT})')

## Cell 9 — Training Curves

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training Progress', fontsize=16, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(epochs_ran, history['train_loss'], 'b-o', ms=3, label='Train Loss')
ax.plot(epochs_ran, history['val_loss'],   'r-o', ms=3, label='Val Loss')
ax.set_title('Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.legend()
ax.grid(True, alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs_ran, history['train_acc'], 'b-o', ms=3, label='Train Acc')
ax.plot(epochs_ran, history['val_acc'],   'r-o', ms=3, label='Val Acc')
ax.set_title('Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy (%)')
ax.legend()
ax.grid(True, alpha=0.3)

# AUC
ax = axes[2]
ax.plot(epochs_ran, history['train_auc'], 'b-o', ms=3, label='Train AUC')
ax.plot(epochs_ran, history['val_auc'],   'r-o', ms=3, label='Val AUC')
ax.axhline(y=best_val_auc, color='g', linestyle='--', alpha=0.7, label=f'Best AUC={best_val_auc:.1f}%')
ax.set_title('ROC AUC')
ax.set_xlabel('Epoch')
ax.set_ylabel('AUC (%)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Saved → {RESULTS_DIR}/training_curves.png')

## Cell 10 — Load Best Model & Evaluate on Test Set

In [ ]:
# Load best checkpoint
ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'✅ Loaded best model from epoch {ckpt["epoch"]} (Val AUC={ckpt["val_auc"]:.2f}%)')

# Run test evaluation
_, test_acc, test_auc, test_labels, test_probs = run_epoch(
    model, test_loader, criterion, optimizer=None
)
test_preds = (test_probs >= 0.5).astype(int)

print(f'\n📋 Test Results')
print('=' * 40)
print(f'   Accuracy : {test_acc:.2f}%')
print(f'   ROC AUC  : {test_auc:.2f}%')
print(f'   Avg Prec : {average_precision_score(test_labels, test_probs)*100:.2f}%')
print('\n' + classification_report(test_labels, test_preds, target_names=['Normal', 'Anomaly']))

## Cell 11 — Confusion Matrix, ROC, PR Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Test Set Evaluation', fontsize=16, fontweight='bold')

# ── Confusion Matrix ──
cm = confusion_matrix(test_labels, test_preds)
ax = axes[0]
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
ax.set_title('Confusion Matrix')
ax.set_xlabel('Predicted')
ax.set_ylabel('True')

# ── ROC Curve ──
fpr, tpr, _ = roc_curve(test_labels, test_probs)
ax = axes[1]
ax.plot(fpr, tpr, 'b-', lw=2, label=f'ROC (AUC={test_auc:.1f}%)')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5, label='Random')
ax.fill_between(fpr, tpr, alpha=0.1, color='blue')
ax.set_title('ROC Curve')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend()
ax.grid(True, alpha=0.3)

# ── Precision-Recall Curve ──
prec, rec, _ = precision_recall_curve(test_labels, test_probs)
ap = average_precision_score(test_labels, test_probs)
ax = axes[2]
ax.plot(rec, prec, 'r-', lw=2, label=f'PR (AP={ap*100:.1f}%)')
ax.fill_between(rec, prec, alpha=0.1, color='red')
ax.set_title('Precision-Recall Curve')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Saved → {RESULTS_DIR}/test_evaluation.png')

## Cell 12 — Per-Category Analysis

In [ ]:
# Add predictions to test_df for per-category analysis
test_df_eval = test_df.copy().reset_index(drop=True)
test_df_eval['prob_anomaly'] = test_probs
test_df_eval['pred_label']   = test_preds
test_df_eval['correct']      = (test_df_eval['pred_label'] == test_df_eval['label']).astype(int)

cat_perf = test_df_eval.groupby('category').agg(
    count=('correct', 'count'),
    accuracy=('correct', 'mean'),
    mean_prob=('prob_anomaly', 'mean')
).reset_index()
cat_perf['accuracy'] = (cat_perf['accuracy'] * 100).round(1)
cat_perf = cat_perf.sort_values('accuracy', ascending=False)

print('📋 Per-Category Test Accuracy:')
print(cat_perf.to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ecc71' if c in NORMAL_FOLDERS else '#e74c3c' for c in cat_perf['category']]
bars = ax.bar(cat_perf['category'], cat_perf['accuracy'], color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5, label='50% baseline')
ax.set_title('Per-Category Accuracy on Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Category')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 110)
ax.tick_params(axis='x', rotation=45)
for bar, val in zip(bars, cat_perf['accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{val:.0f}%', ha='center', fontsize=8, fontweight='bold')
legend_handles = [
    mpatches.Patch(color='#2ecc71', label='Normal'),
    mpatches.Patch(color='#e74c3c', label='Anomaly')
]
ax.legend(handles=legend_handles)
plt.tight_layout()
fig.savefig(f'{RESULTS_DIR}/per_category_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 13 — Inference on a New Video

In [ ]:
def predict_video(npy_path, model, max_frames=MAX_FRAMES, device=DEVICE, threshold=0.5):
    """
    Predict anomaly score for a single .npy feature file.

    Returns:
        dict with keys: prob_normal, prob_anomaly, prediction, label
    """
    model.eval()
    feat = np.load(npy_path).astype(np.float32)   # (T, 768)
    T    = feat.shape[0]

    # Clip / pad
    if T >= max_frames:
        start = (T - max_frames) // 2
        feat  = feat[start : start + max_frames]
    else:
        pad  = np.zeros((max_frames - T, feat.shape[1]), dtype=np.float32)
        feat = np.concatenate([feat, pad], axis=0)

    feat_t = torch.from_numpy(feat).unsqueeze(0).to(device)  # (1, T, 768)
    with torch.no_grad():
        logits = model(feat_t)
        probs  = F.softmax(logits, dim=-1)[0].cpu().numpy()

    result = {
        'file':          os.path.basename(npy_path),
        'prob_normal':   float(probs[0]),
        'prob_anomaly':  float(probs[1]),
        'prediction':    'ANOMALY' if probs[1] >= threshold else 'NORMAL',
        'confidence':    float(max(probs))
    }
    return result


# ── Demo: run on a few test videos ──
print('🔍 Inference on sample test videos:')
print('─' * 75)
print(f'{"File":<40} {"P(Anomaly)":>12} {"Prediction":>12} {"True Label":>12}')
print('─' * 75)

for _, row in test_df.sample(min(8, len(test_df)), random_state=42).iterrows():
    res = predict_video(row['path'], model)
    true_lbl = 'ANOMALY' if row['label'] == 1 else 'NORMAL'
    match = '✅' if res['prediction'] == true_lbl else '❌'
    print(f"{res['file']:<40} {res['prob_anomaly']:>11.3f}  {res['prediction']:>12} {true_lbl:>10}  {match}")

# ── Predict your own file ──
# Uncomment and change path:
# result = predict_video('/path/to/your/video_features.npy', model)
# print(result)

## Cell 14 — Save Training History CSV

In [ ]:
hist_df = pd.DataFrame(history)
hist_df.index = range(1, len(hist_df) + 1)
hist_df.index.name = 'epoch'
hist_path = f'{RESULTS_DIR}/training_history.csv'
hist_df.to_csv(hist_path)
print(f'✅ Training history saved to {hist_path}')
print(hist_df.tail(5).to_string())

print(f'\n🏆 Final Summary')
print('=' * 40)
print(f'   Best Val AUC  : {best_val_auc:.2f}%')
print(f'   Test Accuracy : {test_acc:.2f}%')
print(f'   Test AUC      : {test_auc:.2f}%')
print(f'   Checkpoint    : {CHECKPOINT}')
print(f'   Results dir   : {RESULTS_DIR}/')